In [1]:
import torch
import torch.nn as nn

In [2]:
batch_size = 64
sequence_len = 64
embedding_dim = 32
layer_size = 256
steps = 5000

In [3]:
# input & output initialization
with open("inputs-shakespeare.txt", "r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(chars)}
itoc = dict(enumerate(chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(0, sequence_len, 1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(0, sequence_len, 1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(0, sequence_len, sequence_len)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(0, sequence_len, sequence_len)

In [4]:
# module declaration
class CausalMean(nn.Module):
    def __init__(self, sequence_len):
        super().__init__()
        tril = torch.tril(torch.ones(sequence_len, sequence_len, dtype=torch.bool))
        self.register_buffer("mask", tril, persistent=False)

    def forward(self, input):
        T = input.size(1)
        scores = torch.zeros(T, T)
        scores = scores.masked_fill(~self.mask[:T, :T], float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        return weights @ input

In [5]:
# model declaration
class BagOfWords(nn.Module):
    def __init__(self, sequence_len, vocab_size, embedding_dim, layer_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Embedding(vocab_size, embedding_dim),
            CausalMean(sequence_len),
            nn.Linear(embedding_dim, layer_size),
            nn.ReLU(),
            nn.Linear(layer_size, vocab_size),
        )

    def forward(self, input):
        return self.layers(input)

In [6]:
# model initialization
model = BagOfWords(sequence_len, vocab_size, embedding_dim, layer_size)
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

In [7]:
# training
for step in range(steps):
    optimizer.zero_grad()
    
    batch = torch.randint(0, len(x_train), (batch_size,))
    train_logits = model(x_train[batch])
    train_loss = nn.functional.cross_entropy(train_logits.view(-1, vocab_size), y_train[batch].view(-1))

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits = model(x_val)
            val_loss = nn.functional.cross_entropy(val_logits.view(-1, vocab_size), y_val.view(-1))
            print(f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}")

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.170709609985352, val loss = 4.172178745269775
Step 1000: train loss = 3.023846387863159, val loss = 3.0670785903930664
Step 2000: train loss = 2.994431972503662, val loss = 3.033980131149292
Step 3000: train loss = 3.0032241344451904, val loss = 3.0237436294555664
Step 4000: train loss = 2.9362339973449707, val loss = 3.0119097232818604
Step 4999: train loss = 2.9824674129486084, val loss = 3.00732684135437


In [8]:
# prediction
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)            

    while len(output) < max_output_length:
        logits = model(i)

        P = torch.softmax(logits[0, -1, :], dim=0)
        j = torch.multinomial(P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[ch] for ch in output[-sequence_len:]]], dtype=torch.long)

In [9]:
print(''.join(output))

ROMEO:
itaid stmet r  ste iwssh h ititetlo in ii tooptihu nocu'kotr lh  gb ,grsyahflhe p tsriy , ay dwdsneesn
h ectbdtueheyeeT iocmeyrhs;eru  ne  eicoa.egnan .sn a?d
dhH

R
bDTkI LiA IOSN:oa sdteoAdsato hLh mrtr:t y ol
ahambttoIy y hroue   e
 r ombakegeaensr d emnyFr a, bd 
htosDho yrfdaaxhleamtoaatkt.i s ,,
  f iI mMerrb eoRo:hIludw
dsw aal fyngdr' li&nat ddrd wywpeifo ao ghdh g s
eesoaor wvs,h ee'soy:ais
dTa
umphahT h en bfe urs hhrt e spthsc eaedri$rann  iI  towlraeb y sperhoareienners tae
'ette,  us sswn yplgidrl tdyHbuh
 A, es caoodnsl araloctTrboddfAnetrhne e i,fse !mo
d hiaods  ola fihumv ? let
te ,'rm ta ryhMstWmsaeot ee
BhecGacRI RunrI t bthayriA:tnnitmd e tlr,ay,tl,h.n
  be
r  eQHloo Ceh
oIht enoony o
r;vlsl OUaEARDLAS:as Eoslwiylowre uerfs, kmenffs,i
 s  : et m fkad
sks pd opptaoronr mhTtoi nyayrethh,Gson ooi nneibmth   ieg g.rnsdteml
a
 ,'Hmn oeged;raysw
sna mI stttic soMhuVry OioEANTrOE:oe; ec  h galhee,.ss

T a:o tmto vlwemdt
ade
mo ebfl rpheeo SWyem iroh h auwtrTsayrheyt